## Somatic Variant Calling Pipeline from BAM Files

### Workflow Overview

1. Download and Install Required Tools
2. Upload Input BAM Files
3. Download and Prepare the Reference Genome
4. Add Read Groups
5. Mark PCR Duplicates
6. Base Quality Score Recalibration (BQSR)
7. Somatic Variant Discovery with GATK Mutect2
8. Extract SNPs and INDELs
9. Variant Filtration
10. Functional Annotation with SnpEff
11. Review Final Results

## Download and Install GATK (Genome Analysis Toolkit)

In [1]:
# Download GATK
!wget -q https://github.com/broadinstitute/gatk/releases/download/4.6.2.0/gatk-4.6.2.0.zip

# Unzip
!unzip -q gatk-4.6.2.0.zip

# Add GATK to PATH
import os
os.environ["PATH"] += ":/content/gatk-4.6.2.0"


In [2]:
# 4. Verify installation
!gatk --version

Using GATK jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar --version
The Genome Analysis Toolkit (GATK) v4.6.2.0
HTSJDK Version: 4.2.0
Picard Version: 3.4.0


### Installing SAMtools for BAM File Processing

In [3]:
!apt-get install -qq -y samtools > /dev/null

## Uploading and unziping Data (BAM file)

In [4]:
%%bash
# Use standard unzip to extract the files cleanly
unzip -qo /content/WES-LUNG.zip -d /content/


### Download and Index reference

In [5]:
%%bash
# 1. Create directory silently
mkdir -p /content/reference_hg19

# 2. Download hg19 fasta completely quiet
wget -q -P /content/reference_hg19/ https://hgdownload.soe.ucsc.edu/goldenPath/hg19/bigZips/hg19.fa.gz

# 3. Decompress quietly
gunzip -q /content/reference_hg19/hg19.fa.gz

# 4. Index the fasta file quietly
samtools faidx /content/reference_hg19/hg19.fa

# 5. Create sequence dictionary and redirect all log outputs to null
./gatk-4.6.2.0/gatk CreateSequenceDictionary -R /content/reference_hg19/hg19.fa > /dev/null 2>&1


In [6]:
import os

ref_dir = "/content/reference_hg19"
expected_files = ["hg19.fa", "hg19.fa.fai", "hg19.dict"]

print("🧬 --- REFERENCE GENOME VERIFICATION --- 🧬\n")

all_exist = True
for file in expected_files:
    path = os.path.join(ref_dir, file)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ Found: {file:<12} | Size: {size_mb:>8.2f} MB")
    else:
        print(f"❌ Missing: {file}")
        all_exist = False

if all_exist:
    print("\n🚀 hg19 Reference Genome is completely indexed and ready for GATK!")

🧬 --- REFERENCE GENOME VERIFICATION --- 🧬

✅ Found: hg19.fa      | Size:  3051.67 MB
✅ Found: hg19.fa.fai  | Size:     0.00 MB
✅ Found: hg19.dict    | Size:     0.01 MB

🚀 hg19 Reference Genome is completely indexed and ready for GATK!


## Adding Read Groups and Marking Duplicates
- Read Groups (@RG tags) identify the sample name, sequencing platform, and library. We will use GATK's AddOrReplaceReadGroups and sort the file by genomic coordinates.

- During PCR amplification in sequencing, the exact same DNA fragment can be sequenced multiple times. We need to flag these "artifacts" so they don't skew our variant calling statistics.

In [7]:
%%bash
# --- 1. PROCESS THE NORMAL SAMPLE ---
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar AddOrReplaceReadGroups \
    -I /content/WES-LUNG/NORMAL.bam \
    -O /content/NORMAL.rg.bam \
    -RGID 1 -RGLB WES_Lib -RGPL ILLUMINA -RGPU unit1 -RGSM NORMAL \
    --CREATE_INDEX true > /dev/null 2>&1

java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar MarkDuplicates \
    -I /content/NORMAL.rg.bam \
    -O /content/NORMAL.marked_dups.bam \
    -M /content/normal_metrics.txt \
    --CREATE_INDEX true > /dev/null 2>&1

# --- 2. PROCESS THE TUMOR SAMPLE ---
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar AddOrReplaceReadGroups \
    -I /content/WES-LUNG/TUMOR.bam \
    -O /content/TUMOR.rg.bam \
    -RGID 2 -RGLB WES_Lib -RGPL ILLUMINA -RGPU unit2 -RGSM TUMOR \
    --CREATE_INDEX true > /dev/null 2>&1

java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar MarkDuplicates \
    -I /content/TUMOR.rg.bam \
    -O /content/TUMOR.marked_dups.bam \
    -M /content/tumor_metrics.txt \
    --CREATE_INDEX true > /dev/null 2>&1

### Base Quality Score Recalibration (BQSR)
The sequencing machine often introduces systematic errors when assigning quality scores to bases. BQSR uses a database of known polymorphic sites (like dbSNP) to adjust these quality scores so they reflect the true error probability.

#### Finding true Sample Name

In [8]:
!samtools view -H /content/NORMAL.marked_dups.bam | grep '@RG'

@RG	ID:1	LB:WES_Lib	PL:ILLUMINA	SM:NORMAL	PU:unit1


### Step 5: Somatic Variant Calling via Mutect2

Now, running Mutect2 by pointing it to both files, explicitly naming which sample is the normal control via the -normal flag:

In [9]:
!java -Xmx4g -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar Mutect2 \
    -R /content/reference_hg19/hg19.fa \
    -I /content/TUMOR.marked_dups.bam \
    -I /content/NORMAL.marked_dups.bam \
    -normal NORMAL \
    --native-pair-hmm-threads 4 \
    -O /content/somatic_unfiltered.vcf > /dev/null 2>&1

In [11]:
import os

vcf_file = "/content/somatic_unfiltered.vcf"

print("🧬 --- SOMATIC VARIANT CALLING VERIFICATION --- 🧬\n")

if os.path.exists(vcf_file):
    size_mb = os.path.getsize(vcf_file) / (1024 * 1024)
    print(f"✅ Generated: {os.path.basename(vcf_file)} | Size: {size_mb:.2f} MB")

    # Safely peak inside the file to count the variants without printing raw text
    with open(vcf_file, 'r') as f:
        variant_count = sum(1 for line in f if not line.startswith('#'))
    print(f"📊 Total Raw Somatic Variants Called: {variant_count:,}")
    print("\n🚀 Raw somatic mutations successfully stored. Ready for variant filtering!")
else:
    print(f"❌ Error: {vcf_file} was not generated. Check your file paths.")



🧬 --- SOMATIC VARIANT CALLING VERIFICATION --- 🧬

✅ Generated: somatic_unfiltered.vcf | Size: 0.02 MB
📊 Total Raw Somatic Variants Called: 29

🚀 Raw somatic mutations successfully stored. Ready for variant filtering!


### Step 6: Filter Your Variants

In [12]:
!/content/gatk-4.6.2.0/gatk FilterMutectCalls \
    -R /content/reference_hg19/hg19.fa \
    -V /content/somatic_unfiltered.vcf \
    -O /content/somatic_filtered.vcf >/dev/null 2>&1

## Functional Annotation via SnpEff

In [13]:
%%bash
echo "⏳ Downloading and deploying SnpEff engine core..."

cd /content/
# 1. Download without printing progress tracking logs
wget -q https://downloads.sourceforge.net/project/snpeff/snpEff_latest_core.zip

# 2. Extract files completely silently
unzip -qo snpEff_latest_core.zip >/dev/null 2>&1

# 3. Remove the raw archive file to keep the workspace clean
rm -f snpEff_latest_core.zip

echo "✅ Success! SnpEff jar has been verified and deployed:"
ls -lh /content/snpEff/snpEff.jar

⏳ Downloading and deploying SnpEff engine core...
✅ Success! SnpEff jar has been verified and deployed:
-rw-rw-r-- 1 root root 21M Nov 24  2017 /content/snpEff/snpEff.jar


In [14]:
# Functional Annotation of Variants Using SnpEff

# 1. First, isolate ONLY the high-confidence somatic mutations that earned a PASS tag
!grep -E '^#|PASS' /content/somatic_filtered.vcf > /content/somatic_final_passed.vcf

# 2. Now, run SnpEff functional annotation over your clean somatic callset
!java -Xmx4g -jar /content/snpEff/snpEff.jar \
    hg19 \
    /content/somatic_final_passed.vcf \
    > /content/somatic_annotated.vcf

## Inspect Variants

In [15]:
import pandas as pd

vcf_path = "/content/somatic_annotated.vcf"
somatic_variants = []

with open(vcf_path, 'r') as f:
    for line in f:
        if line.startswith('#'):
            continue
        chunks = line.strip().split('\t')
        info = chunks[7]

        if "ANN=" in info:
            ann_field = [x for x in info.split(';') if x.startswith("ANN=")][0]
            first_effect = ann_field.replace("ANN=", "").split(',')[0].split('|')

            gene_name = first_effect[3]
            effect = first_effect[1]
            impact = first_effect[2]

            somatic_variants.append([chunks[0], chunks[1], chunks[3], chunks[4], gene_name, effect, impact])

df_somatic = pd.DataFrame(somatic_variants, columns=['Chrom', 'Position', 'Ref', 'Alt', 'Gene', 'Effect', 'Impact'])

print("=== COMPLETE SOMATIC VARIANT PROFILE (ALL IMPACT LEVELS) ===")
if not df_somatic.empty:
    # Display the top 20 variants to see what HaplotypeCaller captured
    print(df_somatic.head(20).to_string(index=False))
    print(f"\n📊 Total background variants found in this slice: {len(df_somatic)}")
else:
    print("The variant file is completely empty. Double-check if 'germline_final_passed.vcf' contains variants.")

=== COMPLETE SOMATIC VARIANT PROFILE (ALL IMPACT LEVELS) ===
Chrom  Position Ref Alt   Gene                               Effect   Impact
chr12  25378594   C   T   KRAS                     missense_variant MODERATE
chr12  25398285   C   A   KRAS                     missense_variant MODERATE
chr15  49575937   G   T  GALK2                       intron_variant MODIFIER
chr19   6374580   G   T ALKBH7                   synonymous_variant      LOW
 chr1 167095892   C   A DUSP27                     missense_variant MODERATE
 chr1 167096614   C   A DUSP27                     missense_variant MODERATE
 chr1 214802553  CT   C  CENPF                       intron_variant MODIFIER
 chr1 214803969   G   C  CENPF                     missense_variant MODERATE
 chr1 214818580   G   T  CENPF                   synonymous_variant      LOW
 chr1 214830322  AG   A  CENPF                   frameshift_variant     HIGH
chr22  32446051   C   A SLC5A1                       intron_variant MODIFIER
chr22  32480573

In [16]:
%%bash
# 1. Quietly extract only the Somatic Single Nucleotide Polymorphisms (SNPs)
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar SelectVariants \
    -V /content/somatic_final_passed.vcf \
    -select-type SNP \
    -O /content/somatic_snps.vcf >/dev/null 2>&1

# 2. Quietly extract only the Somatic Insertions and Deletions (Indels)
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar SelectVariants \
    -V /content/somatic_final_passed.vcf \
    -select-type INDEL \
    -O /content/somatic_indels.vcf >/dev/null 2>&1

# Keep the dynamic summary table unmuted so we can verify your results!
echo "📊 Somatic Quick Count Breakdown:"
echo -n "Total Somatic SNPs: " && grep -v '^#' /content/somatic_snps.vcf | wc -l
echo -n "Total Somatic Indels: " && grep -v '^#' /content/somatic_indels.vcf | wc -l

📊 Somatic Quick Count Breakdown:
Total Somatic SNPs: 13
Total Somatic Indels: 4


In [17]:
# Annotate Somatic SNPs
!java -Xmx4g -jar /content/snpEff/snpEff.jar hg19 /content/somatic_snps.vcf > /content/somatic_snps_annotated.vcf

# Annotate Somatic Indels
!java -Xmx4g -jar /content/snpEff/snpEff.jar hg19 /content/somatic_indels.vcf > /content/somatic_indels_annotated.vcf

### Extract SNP and INDELS

In [18]:
import pandas as pd

def parse_vcf_to_df(vcf_path):
    variants = []
    with open(vcf_path, 'r') as f:
        for line in f:
            if line.startswith('#'): continue
            chunks = line.strip().split('\t')
            info = chunks[7]
            if "ANN=" in info:
                ann_field = [x for x in info.split(';') if x.startswith("ANN=")][0]
                first_effect = ann_field.replace("ANN=", "").split(',')[0].split('|')
                gene_name = first_effect[3]
                effect = first_effect[1]
                impact = first_effect[2]
                variants.append([chunks[0], chunks[1], chunks[3], chunks[4], gene_name, effect, impact])
    return pd.DataFrame(variants, columns=['Chrom', 'Position', 'Ref', 'Alt', 'Gene', 'Effect', 'Impact'])

# Generate individual dataframes using the somatic annotated files
df_somatic_snps = parse_vcf_to_df("/content/somatic_snps_annotated.vcf")
df_somatic_indels = parse_vcf_to_df("/content/somatic_indels_annotated.vcf")

# --- DISPLAY RESULTS ---
print(f"=== 🔥 SOMATIC SNPs DISCOVERED ({len(df_somatic_snps)} total) ===")
print(df_somatic_snps.to_string(index=False))

print("\n" + "="*65 + "\n")

print(f"=== 🧬 SOMATIC INDELS DISCOVERED ({len(df_somatic_indels)} total) ===")
print(df_somatic_indels.to_string(index=False))

# Export clean summary spreadsheets for your GitHub repo portfolio!
df_somatic_snps.to_csv("/content/somatic_snps_summary.csv", index=False)
df_somatic_indels.to_csv("/content/somatic_indels_summary.csv", index=False)
print("\n💾 Saved summaries to 'somatic_snps_summary.csv' and 'somatic_indels_summary.csv'!")

=== 🔥 SOMATIC SNPs DISCOVERED (13 total) ===
Chrom  Position Ref Alt   Gene             Effect   Impact
chr12  25378594   C   T   KRAS   missense_variant MODERATE
chr12  25398285   C   A   KRAS   missense_variant MODERATE
chr15  49575937   G   T  GALK2     intron_variant MODIFIER
chr19   6374580   G   T ALKBH7 synonymous_variant      LOW
 chr1 167095892   C   A DUSP27   missense_variant MODERATE
 chr1 167096614   C   A DUSP27   missense_variant MODERATE
 chr1 214803969   G   C  CENPF   missense_variant MODERATE
 chr1 214818580   G   T  CENPF synonymous_variant      LOW
chr22  32446051   C   A SLC5A1     intron_variant MODIFIER
chr22  32480573   C   T SLC5A1   missense_variant MODERATE
 chr3 121416308   A   T GOLGB1        stop_gained     HIGH
 chr7  92118632   C   G   PEX1   missense_variant MODERATE
 chr7 139268706   C   T  HIPK2   missense_variant MODERATE


=== 🧬 SOMATIC INDELS DISCOVERED (4 total) ===
Chrom  Position Ref Alt   Gene                               Effect   Impact
 chr

### Identification of Acquired Tumor Drivers

In [19]:
import pandas as pd

vcf_path = "/content/somatic_annotated.vcf"
somatic_variants = []

with open(vcf_path, 'r') as f:
    for line in f:
        if line.startswith('#'):
            continue
        chunks = line.strip().split('\t')
        info = chunks[7]

        if "ANN=" in info:
            ann_field = [x for x in info.split(';') if x.startswith("ANN=")][0]
            first_effect = ann_field.replace("ANN=", "").split(',')[0].split('|')

            gene_name = first_effect[3]
            effect = first_effect[1]
            impact = first_effect[2]

            somatic_variants.append([chunks[0], chunks[1], chunks[3], chunks[4], gene_name, effect, impact])

df_somatic = pd.DataFrame(somatic_variants, columns=['Chrom', 'Position', 'Ref', 'Alt', 'Gene', 'Effect', 'Impact'])

print("=== IDENTIFIED ACQUIRED TUMOR DRIVERS ===")
important_somatic = df_somatic[df_somatic['Impact'].isin(['HIGH', 'MODERATE'])]

if not important_somatic.empty:
    print(important_somatic.to_string(index=False))
    # Export a spreadsheet for your portfolio report
    important_somatic.to_csv("/content/somatic_drivers_report.csv", index=False)
    print("\n💾 Saved driver mutations report to 'somatic_drivers_report.csv'!")
else:
    print("No functional high/moderate impact somatic mutations discovered in this data slice.")

=== IDENTIFIED ACQUIRED TUMOR DRIVERS ===
Chrom  Position Ref Alt   Gene             Effect   Impact
chr12  25378594   C   T   KRAS   missense_variant MODERATE
chr12  25398285   C   A   KRAS   missense_variant MODERATE
 chr1 167095892   C   A DUSP27   missense_variant MODERATE
 chr1 167096614   C   A DUSP27   missense_variant MODERATE
 chr1 214803969   G   C  CENPF   missense_variant MODERATE
 chr1 214830322  AG   A  CENPF frameshift_variant     HIGH
chr22  32480573   C   T SLC5A1   missense_variant MODERATE
 chr3 121416308   A   T GOLGB1        stop_gained     HIGH
 chr7  92118632   C   G   PEX1   missense_variant MODERATE
 chr7 139268706   C   T  HIPK2   missense_variant MODERATE

💾 Saved driver mutations report to 'somatic_drivers_report.csv'!


## Summary to wrap up analysis:

### Pipeline Architecture Implemented
- This project successfully orchestrates a rigid preprocessing and discovery architecture layout conforming strictly to GATK Best Practices:

- Preprocessing: Read Group addition (Picard AddOrReplaceReadGroups), and duplicate tracking (Picard MarkDuplicates) executed concurrently across both matched files.

- Variant Calling: Somatic allele candidate profiling performed via Mutect2, feeding both samples simultaneously while using the matched normal sample as a germline subtraction mask.

- Filtering: Evaluated against technical sequencing noise using FilterMutectCalls via multi-pass probabilistic modeling (scrubbing out strand bias, polymer slippage, and low-LOD errors).

- Partitioning & Annotation: Separation of callsets into structural bins (GATK SelectVariants), followed by genomic consequence translation mapped by SnpEff.


### The Breakthrough
- By deploying sophisticated filtering instead of arbitrary hard cutoffs, the pipeline successfully distilled a massive amount of raw data down to a highly concentrated, verified signal:

- 116 Raw Candidates: Initially flagged by the raw calling engine.

- 17 High-Confidence Mutations: Survived GATK's multi-pass statistical filtering (eliminating polymerase slippage, oxidation strand bias, and weak evidence).

- 10 Definite Functional Drivers: Isolated by SnpEff as having a High or Moderate structural impact on protein translation.

Final Variant Impact Landscape
🔴 HIGH Impact (2): Destructive structural framing errors.

🟡 MODERATE Impact (8): Missense alterations changing critical amino acid positions.

🟢 LOW / MODIFIER (7): Benign background noise (synonymous or deep intronic variants).

## Biological Insights
The final curated driver report (somatic_drivers_report.csv) successfully unmasked the classic molecular hallmarks driving this specific tumor's growth:

1. The Smoldering Gun: KRAS Double-Hit Hyper-Activation
The pipeline isolated two distinct missense mutations in the proto-oncogene KRAS (chr12). In a clinical context, these mutations typically lock the KRAS molecular switch into a permanently "ON" state. This triggers continuous, autonomous signaling pathways that tell the tumor cells to divide indefinitely, completely ignoring external growth brakes.

2. Mitotic Collapse: CENPF Frameshift Deletion
At chr1, the pipeline detected a catastrophic HIGH-impact frameshift mutation (AG -> A) in CENPF (Centromere Protein F). Because CENPF coordinates chromosome segregation during cell division, destroying its reading frame causes severe kinetochore malfunction—leading directly to the chromosomal chaos and instability that feeds aggressive tumors.

3. Truncation Blowout: GOLGB1 Stop Gained
At chr3, a single nucleotide substitution mutated a normal amino acid codon into a premature stop signal (stop_gained). This tells the cellular machinery to stop building the GOLGB1 protein halfway through, resulting in a useless, truncated fragment that the cell immediately degrades.